In [ ]:
!pip uninstall -y scipy numpy
!pip install -q numpy==1.26.4
!pip install -q scipy==1.13.1
!pip install -q albumentations==1.3.1
!pip install -q scikit-image==0.21.0

print(" Dependencies installed ")

In [ ]:

# imports

print("\nCELL 2: Importing libraries...")
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import json
from datetime import datetime
import shutil
import math
import gc
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
from sklearn.model_selection import train_test_split

plt.style.use('default')
plt.rcParams['figure.dpi'] = 150 #

print("All libraries imported successfully")
print(f"NumPy version: {np.__version__}")
print(f"PyTorch version: {torch.__version__}")

In [ ]:

# Configuration

print("\n Setting up configuration for HYBRID L1-Loss RUN...")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"\n{'='*70}")
print(f" DDPM CONFIGURATION (Hybrid L1-Loss Run, 150 Epochs - LR=2e-4)")
print(f"{'='*70}")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"{'='*70}\n")

def set_seed(seed=42):
    import random
    import numpy as np
    import torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

class Config:
   
    input_data_root = "/kaggle/input/mri-ct/content/drive/MyDrive/synthRAD2023/processed"
    data_root = "/kaggle/working/data_full"
    output_dir = "/kaggle/working/outputs_HYBRID_loss_run"
    resume_checkpoint = "/kaggle/input/last-checkpoint/checkpoint_epoch_140_full (1).pt"
    start_epoch = 1

    img_size = 256
    batch_size = 8
    num_workers = 4
    max_train_samples = 28747
    max_val_samples = 2000
    val_split = max_val_samples / (max_train_samples + max_val_samples)
    
    epochs = 160
    base_channels = 56
    channel_multipliers = (1, 2, 4, 6)
    num_res_blocks = 2
    attention_resolutions = (32, 16,)
    dropout = 0.1

    timesteps = 1000
    beta_start = 0.0001
    beta_end = 0.02
    learning_rate = 2e-4
    
    weight_decay = 0.0
    grad_clip = 1.0
    warmup_epochs = 5

    use_amp = True
    use_ema = True
    ema_decay = 0.9999
    save_interval = 10
    sample_interval = 5
    val_interval = 5
    num_sample_images = 4

    ddim_steps = 50
    eval_ddim_steps = 50

    device = device

config = Config()

# Create directories
os.makedirs(config.output_dir, exist_ok=True)
os.makedirs(os.path.join(config.output_dir, 'checkpoints'), exist_ok=True)
os.makedirs(os.path.join(config.output_dir, 'samples'), exist_ok=True)
os.makedirs(os.path.join(config.output_dir, 'figures'), exist_ok=True)

print(" CONFIGURATION SUMMARY (Hybrid L1 Loss Run - From Scratch)")
print(f"\n[TRAINING PLAN - HYBRID L1 LOSS]")
print(f"  Samples: {config.max_train_samples} train + {config.max_val_samples} val")
print(f"  Epochs: {config.epochs}")
print(f"  Learning Rate: {config.learning_rate}")
print(f"  Attention Res: {config.attention_resolutions}")
print(f"  Resume from: None. Training from scratch.")

import json
config_dict = {k: str(v) for k, v in vars(config).items() if not k.startswith('_')}
with open(os.path.join(config.output_dir, 'config_HYBRID_loss_run.json'), 'w') as f:
    json.dump(config_dict, f, indent=4)
print(" Config saved.")

In [ ]:


print("\nChecking .npy file value ranges...")
input_mri_dir_check = os.path.join(config.input_data_root, 'mri_slices')
input_ct_dir_check = os.path.join(config.input_data_root, 'ct_slices')

try:
    mri_files_check = sorted([f for f in os.listdir(input_mri_dir_check) if f.endswith('.npy')])
    ct_files_check = sorted([f for f in os.listdir(input_ct_dir_check) if f.endswith('.npy')])

    if not mri_files_check:
        print(f"ERROR: No .npy files found in MRI directory: {input_mri_dir_check}")
    if not ct_files_check:
        print(f"ERROR: No .npy files found in CT directory: {input_ct_dir_check}")

    if mri_files_check and ct_files_check:
        mri_sample_path = os.path.join(input_mri_dir_check, mri_files_check[0])
        ct_sample_path = os.path.join(input_ct_dir_check, ct_files_check[0]) 

        mri_sample = np.load(mri_sample_path)
        ct_sample = np.load(ct_sample_path)

        print(f"\nAnalysis for Sample Files ")
        print(f"MRI Sample ({mri_files_check[0]}):")
        print(f"  Shape: {mri_sample.shape}")
        print(f"  Data Type (dtype): {mri_sample.dtype}")
        print(f"  Minimum Value: {mri_sample.min():.4f}")
        print(f"  Maximum Value: {mri_sample.max():.4f}")
        print(f"  Mean Value: {mri_sample.mean():.4f}")

        print(f"\nCT Sample ({ct_files_check[0]}):")
        print(f"  Shape: {ct_sample.shape}")
        print(f"  Data Type (dtype): {ct_sample.dtype}")
        print(f"  Minimum Value: {ct_sample.min():.4f}")
        print(f"  Maximum Value: {ct_sample.max():.4f}")
        print(f"  Mean Value: {ct_sample.mean():.4f}")
        print(f" End Analysis")

        print("\n INTERPRETATION")
        print(" - If Max values are near 1.0 (and dtype is float), REMOVE A.Normalize")
        print(" - If Max values are near 255 (and dtype is int/uint8), USE A.Normalize")
        print(" - If ranges are different, custom scaling might be needed.")

except FileNotFoundError:
    print(f"ERROR: Could not find input directories:")
    print(f"  MRI: {input_mri_dir_check}")
    print(f"  CT: {input_ct_dir_check}")
    print("Please double-check `config.input_data_root`.")
except Exception as e:
    print(f"An error occurred while checking .npy files: {e}")

In [ ]:

# Data Prepartion

print("\nPreparing data (Full Dataset), defining Dataset, creating Dataloaders")

import os
import shutil
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

input_mri_dir = os.path.join(config.input_data_root, 'mri_slices')
input_ct_dir = os.path.join(config.input_data_root, 'ct_slices')
for split in ['train', 'val']:
    os.makedirs(os.path.join(config.data_root, 'mri_slices', split), exist_ok=True)
    os.makedirs(os.path.join(config.data_root, 'ct_slices', split), exist_ok=True)
all_files = []
if os.path.exists(input_mri_dir):
    all_files = sorted([f for f in os.listdir(input_mri_dir) if f.endswith('.npy') and not f.startswith('.')])
else: print(f"ERROR: Input MRI directory not found at {input_mri_dir}.")
if all_files:
    print(f"Total available files found: {len(all_files)}")
    train_files, val_files = train_test_split(all_files, test_size=config.val_split, random_state=42)
    train_files = train_files[:config.max_train_samples]
    val_files = val_files[:config.max_val_samples]
    print(f"Using: {len(train_files)} train, {len(val_files)} val")
    print("\nCopying files to working directory (this may take a while)")
    copy_error = False
    for f in tqdm(train_files, desc='Train Copy'):
        try:
            shutil.copy(os.path.join(input_mri_dir, f), os.path.join(config.data_root, 'mri_slices', 'train', f))
            shutil.copy(os.path.join(input_ct_dir, f), os.path.join(config.data_root, 'ct_slices', 'train', f))
        except FileNotFoundError: print(f"Error copying file {f}."); copy_error = True; break
    if not copy_error:
        for f in tqdm(val_files, desc='Val Copy'):
            try:
                shutil.copy(os.path.join(input_mri_dir, f), os.path.join(config.data_root, 'mri_slices', 'val', f))
                shutil.copy(os.path.join(input_ct_dir, f), os.path.join(config.data_root, 'ct_slices', 'val', f))
            except FileNotFoundError: print(f"Error copying file {f}."); copy_error = True; break
    if not copy_error: print(f" Data copied successfully!")
    else: print(f" Errors occurred during file copying.")
else: print(f" No files found to copy.")

class MRICTDataset(Dataset):
    def __init__(self, mri_dir, ct_dir, transform=None, img_size=256):
        self.mri_dir = mri_dir
        self.ct_dir = ct_dir
        self.img_size = img_size
        self.transform = transform
        self.mri_files = sorted([f for f in os.listdir(mri_dir) if f.endswith('.npy')])
        self.ct_files = sorted([f for f in os.listdir(ct_dir) if f.endswith('.npy')])
        assert len(self.mri_files) == len(self.ct_files), "Mismatch between MRI and CT file counts!"
        self.print_debug = True 

    def __len__(self):
        return len(self.mri_files)

    def __getitem__(self, idx):
        mri_path = os.path.join(self.mri_dir, self.mri_files[idx])
        ct_path = os.path.join(self.ct_dir, self.ct_files[idx])
        try:
            mri = np.load(mri_path).astype(np.float32)
            ct = np.load(ct_path).astype(np.float32)
        except Exception as e:
            print(f"Warning: Error loading file index {idx}: {e}")
            return {'mri': torch.zeros((1, self.img_size, self.img_size)), 'ct': torch.zeros((1, self.img_size, self.img_size)), 'filename': 'error'}

        if idx == 0 and self.print_debug:
            print(f"\n DEBUG: Data Loading (Index {idx})")
            print(f"Loaded .npy - MRI min: {mri.min():.4f}, max: {mri.max():.4f}, shape: {mri.shape}, dtype: {mri.dtype}")
            print(f"Loaded .npy - CT  min: {ct.min():.4f}, max: {ct.max():.4f}, shape: {ct.shape}, dtype: {ct.dtype}")
     

        mri_tensor, ct_tensor = None, None
        if self.transform:
            try:
                transformed = self.transform(image=mri, mask=ct)
                mri_tensor = transformed['image']
                ct_tensor = transformed['mask']
            except Exception as e:
                print(f"Warning: Error during transform for index {idx}: {e}")
                return {'mri': torch.zeros((1, self.img_size, self.img_size)), 'ct': torch.zeros((1, self.img_size, self.img_size)), 'filename': 'error'}

            if idx == 0 and self.print_debug:
                print(f"After Albumentations - MRI min: {mri_tensor.min():.4f}, max: {mri_tensor.max():.4f}, shape: {mri_tensor.shape}, dtype: {mri_tensor.dtype}")
                print(f"After Albumentations - CT  min: {ct_tensor.min():.4f}, max: {ct_tensor.max():.4f}, shape: {ct_tensor.shape}, dtype: {ct_tensor.dtype}")
                print(f" END DEBUG \n")
                self.print_debug = False 
          
        if mri_tensor is not None and len(mri_tensor.shape) == 2: mri_tensor = mri_tensor.unsqueeze(0)
        if ct_tensor is not None and len(ct_tensor.shape) == 2: ct_tensor = ct_tensor.unsqueeze(0)
        if mri_tensor is None or ct_tensor is None:
             print(f"Warning: Tensor creation failed for index {idx}, returning zeros.")
             mri_tensor = torch.zeros((1, self.img_size, self.img_size))
             ct_tensor = torch.zeros((1, self.img_size, self.img_size))

        return {'mri': mri_tensor.float(), 'ct': ct_tensor.float(), 'filename': self.mri_files[idx]}


def get_dataloaders(data_root, batch_size, img_size, num_workers):
    train_mri_dir = os.path.join(data_root, 'mri_slices', 'train')
    train_ct_dir = os.path.join(data_root, 'ct_slices', 'train')
    val_mri_dir = os.path.join(data_root, 'mri_slices', 'val')
    val_ct_dir = os.path.join(data_root, 'ct_slices', 'val')

    if not os.path.exists(train_mri_dir) or not os.listdir(train_mri_dir):
        print(f"ERROR: Training MRI directory is empty or missing: {train_mri_dir}")
        return None, None
    if not os.path.exists(val_mri_dir) or not os.listdir(val_mri_dir):
        print(f"ERROR: Validation MRI directory is empty or missing: {val_mri_dir}")
        return None, None

    print("Using Geometric + Intensity Augmentations")
    
    train_transform = A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=15, p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.3),
        A.GaussNoise(var_limit=(0.0, 0.01), p=0.2),
        ToTensorV2() 
    ])

    val_transform = A.Compose([
        A.Resize(img_size, img_size),
        ToTensorV2() 
    ])

    try:
        train_dataset = MRICTDataset(train_mri_dir, train_ct_dir, train_transform, img_size)
        val_dataset = MRICTDataset(val_mri_dir, val_ct_dir, val_transform, img_size)
    except AssertionError as e:
        print(f"ERROR creating dataset: {e}"); return None, None
    except Exception as e:
        print(f"Unexpected error during dataset creation: {e}"); return None, None

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                               num_workers=num_workers, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                               num_workers=num_workers, pin_memory=True)

    return train_loader, val_loader

print("Creating dataloaders with full dataset and intensity augmentations...")
train_loader, val_loader = get_dataloaders(
    config.data_root, config.batch_size, config.img_size, config.num_workers
)

if train_loader: print(f" Train batches: {len(train_loader)}")
if val_loader: print(f" Val batches: {len(val_loader)}")

In [ ]:

# Visualization

print("\n Visualizing a sample batch")
if train_loader:
    try:
        batch = next(iter(train_loader))
        mri_batch = batch['mri']
        ct_batch = batch['ct']

        print(f"Batch MRI shape: {mri_batch.shape}, dtype: {mri_batch.dtype}, range: [{mri_batch.min():.2f}, {mri_batch.max():.2f}]")
        print(f"Batch CT shape: {ct_batch.shape}, dtype: {ct_batch.dtype}, range: [{ct_batch.min():.2f}, {ct_batch.max():.2f}]")

        if mri_batch.min() < -1.1 or mri_batch.max() > 1.1 or ct_batch.min() < -1.1 or ct_batch.max() > 1.1:
             print("WARNING: Tensor values seem outside the expected [-1, 1] range!")
        else:
             print("Tensor value range looks correct (approx [-1, 1]).") 

        n_display = min(4, config.batch_size)
        fig, axes = plt.subplots(n_display, 2, figsize=(6, 2 * n_display))
        if n_display == 1: axes = axes.reshape(1, -1)

        for i in range(n_display):
            mri_img = (mri_batch[i, 0].cpu().numpy() + 1) / 2
            ct_img = (ct_batch[i, 0].cpu().numpy() + 1) / 2

            axes[i, 0].imshow(mri_img, cmap='gray', vmin=0, vmax=1)
            axes[i, 0].set_title(f'MRI Input (Sample {i})')
            axes[i, 0].axis('off')

            axes[i, 1].imshow(ct_img, cmap='gray', vmin=0, vmax=1)
            axes[i, 1].set_title(f'CT Target (Sample {i})')
            axes[i, 1].axis('off')
        plt.tight_layout()
        plt.suptitle("Sample Input Batch (Should be [-1, 1] Tensors)", y=1.03)
        plt.savefig(os.path.join(config.output_dir, 'figures', 'sample_input_batch_corrected.png'), dpi=100)
        plt.close()

    except StopIteration:
        print("ERROR: Could not get a batch from train_loader. Is the dataset empty?")
    except Exception as e:
        print(f"Error visualizing batch: {e}")
else:
    print("Skipping batch visualization because train_loader was not created.")
print(" End Batch Visualization")

In [ ]:

# Model Components

print("\nDefining model components and UNet")

class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        if self.dim % 2 == 1: # Pad if dim is odd
             embeddings = F.pad(embeddings, (0,1))
        return embeddings

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, time_dim, dropout=0.1):
        super().__init__()
        self.time_mlp = nn.Linear(time_dim, out_channels)
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.norm1 = nn.GroupNorm(8, out_channels)
        self.norm2 = nn.GroupNorm(8, out_channels)
        self.dropout = nn.Dropout(dropout)
        self.residual_conv = nn.Conv2d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x, t):
        h = self.conv1(x)
        h = self.norm1(h)
        h = h + self.time_mlp(t)[:, :, None, None]
        h = F.silu(h)
        h = self.dropout(h)
        h = self.conv2(h)
        h = self.norm2(h)
        h = F.silu(h)
        return h + self.residual_conv(x)

class AttentionBlock(nn.Module):
    def __init__(self, channels, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.norm = nn.GroupNorm(8, channels)
        self.qkv = nn.Conv2d(channels, channels * 3, 1)
        self.proj = nn.Conv2d(channels, channels, 1)

    def forward(self, x):
        b, c, h, w = x.shape
        x_norm = self.norm(x)
        qkv = self.qkv(x_norm)
        q, k, v = qkv.chunk(3, dim=1)
        q = q.view(b, self.num_heads, c // self.num_heads, h * w).transpose(2, 3)
        k = k.view(b, self.num_heads, c // self.num_heads, h * w).transpose(2, 3)
        v = v.view(b, self.num_heads, c // self.num_heads, h * w).transpose(2, 3)
        attn = torch.softmax(q @ k.transpose(-2, -1) / math.sqrt(c // self.num_heads), dim=-1)
        out = (attn @ v).transpose(2, 3).reshape(b, c, h, w)
        out = self.proj(out)
        return out + x

class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1, condition_channels=1,
                 base_channels=56, channel_multipliers=(1, 2, 4, 6),
                 num_res_blocks=2, attention_resolutions=(16,), dropout=0.1):
        super().__init__()
        self.in_channels = in_channels + condition_channels
        time_dim = base_channels * 4
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(base_channels),
            nn.Linear(base_channels, time_dim), nn.SiLU(), nn.Linear(time_dim, time_dim)
        )
        channels = [base_channels] + [base_channels * m for m in channel_multipliers]
        num_levels = len(channels)
        self.input_proj = nn.Conv2d(self.in_channels, base_channels, 3, padding=1)
        self.encoder_blocks = nn.ModuleList()
        self.downsample_blocks = nn.ModuleList()
        in_ch = base_channels
        current_res = config.img_size
        for i in range(num_levels):
            out_ch = channels[i]
            blocks = nn.ModuleList()
            for _ in range(num_res_blocks):
                blocks.append(ResidualBlock(in_ch, out_ch, time_dim, dropout))
                in_ch = out_ch
                if current_res in attention_resolutions:
                    blocks.append(AttentionBlock(out_ch))
            self.encoder_blocks.append(blocks)
            self.downsample_blocks.append(nn.Conv2d(out_ch, out_ch, 3, stride=2, padding=1) if i < num_levels - 1 else nn.Identity())
            if i < num_levels - 1: current_res //= 2
        mid_ch = channels[-1]
        self.middle = nn.ModuleList([
            ResidualBlock(mid_ch, mid_ch, time_dim, dropout), AttentionBlock(mid_ch),
            ResidualBlock(mid_ch, mid_ch, time_dim, dropout)
        ])
        self.decoder_blocks = nn.ModuleList()
        self.upsample_blocks = nn.ModuleList()
        in_ch = mid_ch
        for i in reversed(range(num_levels)):
            out_ch = channels[i]
            skip_ch = channels[i]
            blocks = nn.ModuleList()
            res_input_ch = in_ch + skip_ch
            blocks.append(ResidualBlock(res_input_ch, out_ch, time_dim, dropout))
            in_ch = out_ch
            current_res *= 2
            if current_res in attention_resolutions: blocks.append(AttentionBlock(out_ch))
            for _ in range(num_res_blocks - 1):
                blocks.append(ResidualBlock(in_ch, out_ch, time_dim, dropout))
                if current_res in attention_resolutions: blocks.append(AttentionBlock(out_ch))
            self.decoder_blocks.append(blocks)
            self.upsample_blocks.append(nn.ConvTranspose2d(out_ch, channels[i-1], 4, stride=2, padding=1) if i > 0 else nn.Identity())
            if i > 0: in_ch = channels[i-1]
        self.final = nn.Sequential(nn.GroupNorm(8, base_channels), nn.SiLU(), nn.Conv2d(base_channels, out_channels, 3, padding=1))

    def forward(self, x, timesteps, condition):
        x = torch.cat([x, condition], dim=1)
        t = self.time_mlp(timesteps)
        x = self.input_proj(x)
        skip_connections = []
        for level_blocks, downsample in zip(self.encoder_blocks, self.downsample_blocks):
            for module in level_blocks:
                x = module(x, t) if isinstance(module, ResidualBlock) else module(x)
            skip_connections.append(x)
            x = downsample(x)
        for module in self.middle:
            x = module(x, t) if isinstance(module, ResidualBlock) else module(x)
        skip_connections = list(reversed(skip_connections))
        for i, (level_blocks, upsample) in enumerate(zip(self.decoder_blocks, self.upsample_blocks)):
            skip = skip_connections[i]
            if x.shape[-2:] != skip.shape[-2:]: x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=False)
            x = torch.cat([x, skip], dim=1)
            for module in level_blocks:
                 x = module(x, t) if isinstance(module, ResidualBlock) else module(x)
            x = upsample(x)
        return self.final(x)

print("Building model")
model = UNet(
    in_channels=1, out_channels=1, condition_channels=1,
    base_channels=config.base_channels, channel_multipliers=config.channel_multipliers,
    num_res_blocks=config.num_res_blocks, attention_resolutions=config.attention_resolutions,
    dropout=config.dropout
).to(config.device)

total_params = sum(p.numel() for p in model.parameters())
print(f" Model: {total_params:,} params ({total_params/1e6:.1f}M)")

# Test forward pass
print("\nTesting model forward pass")
try:
    test_noise = torch.randn(2, 1, config.img_size, config.img_size).to(config.device)
    test_condition = torch.randn(2, 1, config.img_size, config.img_size).to(config.device)
    test_t = torch.randint(0, config.timesteps, (2,)).to(config.device)
    with torch.no_grad():
        test_output = model(test_noise, test_t, test_condition)
        print(f" Test passed! Output shape: {test_output.shape}")
        assert test_output.shape == test_noise.shape, "Output shape mismatch"
except Exception as e:
    print(f"ERROR during model test forward pass: {e}")

torch.cuda.empty_cache()
print(" Model ready.")

In [ ]:

# Diffusion Process

print("\nDefining Diffusion Process (HYBRID L1-Loss)")

class DDPMDiffusion:
    def __init__(self, model, timesteps=1000, beta_start=0.0001, beta_end=0.02, device='cuda'):
        self.model = model
        self.timesteps = timesteps
        self.device = device
        self.betas = torch.linspace(beta_start, beta_end, timesteps, device=device)
        self.alphas = 1.0 - self.betas
        self.alphas_cumprod = torch.cumprod(self.alphas, dim=0)
        self.alphas_cumprod_prev = F.pad(self.alphas_cumprod[:-1], (1, 0), value=1.0)
        self.sqrt_alphas_cumprod = torch.sqrt(self.alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - self.alphas_cumprod)
        self.posterior_variance = self.betas * (1.0 - self.alphas_cumprod_prev) / (1.0 - self.alphas_cumprod)
        self.sqrt_recip_alphas_cumprod = torch.sqrt(1.0 / self.alphas_cumprod)
        self.sqrt_recipm1_alphas_cumprod = torch.sqrt(1.0 / self.alphas_cumprod - 1)

    def q_sample(self, x_start, t, noise=None):
        noise = torch.randn_like(x_start) if noise is None else noise
        sqrt_alphas_cumprod_t = self.sqrt_alphas_cumprod.gather(0, t).reshape(-1, 1, 1, 1)
        sqrt_one_minus_alphas_cumprod_t = self.sqrt_one_minus_alphas_cumprod.gather(0, t).reshape(-1, 1, 1, 1)
        return sqrt_alphas_cumprod_t * x_start + sqrt_one_minus_alphas_cumprod_t * noise

    def predict_x0_from_noise(self, x_t, t, noise):
        sqrt_recip_alphas_cumprod_t = self.sqrt_recip_alphas_cumprod.gather(0, t).reshape(-1, 1, 1, 1)
        sqrt_recipm1_alphas_cumprod_t = self.sqrt_recipm1_alphas_cumprod.gather(0, t).reshape(-1, 1, 1, 1)
        return sqrt_recip_alphas_cumprod_t * x_t - sqrt_recipm1_alphas_cumprod_t * noise

    def p_losses(self, x_start, condition, t, noise=None):
        """ Loss calculation - L1(noise) + L1(image) """
        if noise is None: noise = torch.randn_like(x_start)
        
        x_noisy = self.q_sample(x_start, t, noise)
        predicted_noise = self.model(x_noisy, t, condition)
        
        loss_noise = F.l1_loss(predicted_noise, noise)
        x_recons = self.predict_x0_from_noise(x_noisy, t, predicted_noise)
        loss_image = F.l1_loss(x_recons, x_start)
        total_loss = loss_noise + loss_image
        return total_loss, loss_noise, loss_image
    
    @torch.no_grad()
    def ddim_sample_step(self, x_t, t, t_prev, condition, eta=0.0):
        t_tensor = torch.full((x_t.shape[0],), t, device=self.device, dtype=torch.long)
        pred_noise = self.model(x_t, t_tensor, condition)
        alpha_t = self.alphas_cumprod[t].clone().detach()
        alpha_t_prev = self.alphas_cumprod[t_prev].clone().detach() if t_prev >= 0 else torch.tensor(1.0, device=self.device)
        pred_x0 = self.predict_x0_from_noise(x_t, t_tensor, pred_noise).clamp(-1., 1.)
        sigma = 0.0
        if t > t_prev and eta > 0:
            variance_term = torch.clamp((1.0 - alpha_t_prev) / (1.0 - alpha_t) * (1.0 - alpha_t / alpha_t_prev), min=1e-12)
            sigma = eta * torch.sqrt(variance_term)
        sigma_sq = sigma**2 if isinstance(sigma, torch.Tensor) else torch.tensor(sigma**2, device=self.device)
        sqrt_one_minus_alpha_t_prev_minus_sigma_sq_term = torch.clamp(1.0 - alpha_t_prev - sigma_sq, min=1e-12)
        sqrt_one_minus_alpha_t_prev_minus_sigma_sq = torch.sqrt(sqrt_one_minus_alpha_t_prev_minus_sigma_sq_term)
        pred_dir_xt = sqrt_one_minus_alpha_t_prev_minus_sigma_sq * pred_noise
        noise_component = torch.randn_like(x_t) * sigma if t > 0 and eta > 0 else torch.zeros_like(x_t)
        x_prev = torch.sqrt(alpha_t_prev) * pred_x0 + pred_dir_xt + noise_component
        return x_prev

    @torch.no_grad()
    def ddim_sample(self, condition, num_steps=50, eta=0.0):
        self.model.eval()
        b, _, h, w = condition.shape
        timesteps_np = np.linspace(self.timesteps - 1, 0, num_steps, dtype=int)
        timesteps = torch.from_numpy(timesteps_np).long().to(self.device)
        timesteps_prev_np = np.concatenate([timesteps_np[1:], [-1]])
        timesteps_prev = torch.from_numpy(timesteps_prev_np).long().to(self.device)
        img = torch.randn(b, 1, h, w, device=self.device)
        for t, t_prev in tqdm(zip(timesteps, timesteps_prev), total=num_steps, desc='DDIM Sampling', leave=False):
            img = self.ddim_sample_step(img, t, t_prev, condition, eta)
        self.model.train()
        return img

diffusion = DDPMDiffusion(model, config.timesteps, config.beta_start, config.beta_end, config.device)
print("Diffusion process ready (HYBRID L1 Loss).")

In [ ]:

# Ema

print("\n Setting up EMA")
class EMA:
    def __init__(self, model, decay=0.9999):
        self.model = model
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        self.register()

    def register(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()
        print(f" EMA initialized with {len(self.shadow)} parameters.")

    def update(self):
        if not self.shadow: self.register()
        for name, param in self.model.named_parameters():
            if param.requires_grad and name in self.shadow:
                new_average = (1.0 - self.decay) * param.data + self.decay * self.shadow[name]
                self.shadow[name] = new_average.clone()

    def apply_shadow(self):
        self.backup = {}
        for name, param in self.model.named_parameters():
            if param.requires_grad and name in self.shadow:
                self.backup[name] = param.data.clone()
                param.data = self.shadow[name]

    def restore(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and name in self.backup:
                param.data = self.backup[name]
        self.backup = {}

ema = EMA(model, config.ema_decay) if config.use_ema else None
if not config.use_ema: print(" EMA is disabled.")

In [ ]:

# Validation

print("\nDefining Validation Function (Tracking all losses)")
@torch.no_grad()
def validate_epoch(model, diffusion_obj, val_loader, epoch):
    if not val_loader:
        print("Skipping validation: No validation loader.")
        return float('inf'), 0.0, 0.0, float('inf'), float('inf')

    model.eval()
    val_loss_total, val_l1_total, val_percep_total = 0.0, 0.0, 0.0
    psnr_list, ssim_list = [], []

    try:
        batch = next(iter(val_loader))
        mri_sample = batch['mri'][:config.num_sample_images].to(config.device)
        ct_real_sample = batch['ct'][:config.num_sample_images].to(config.device)
        ct_gen_sample = diffusion_obj.ddim_sample(mri_sample, num_steps=config.eval_ddim_steps)

        for i in range(config.num_sample_images):
            real_np = ((ct_real_sample[i, 0].cpu().numpy() + 1) / 2).clip(0, 1)
            gen_np = ((ct_gen_sample[i, 0].cpu().numpy() + 1) / 2).clip(0, 1)
            try:
                psnr_val = psnr(real_np, gen_np, data_range=1.0)
                ssim_val = ssim(real_np, gen_np, data_range=1.0, channel_axis=None, win_size=7)
            except ValueError as e:
                 print(f"Warning: Metric calculation error (sample {i}): {e}. Setting to 0.")
                 psnr_val, ssim_val = 0.0, 0.0
            psnr_list.append(psnr_val)
            ssim_list.append(ssim_val)

        for batch in val_loader:
            mri = batch['mri'].to(config.device)
            ct = batch['ct'].to(config.device)
            t = torch.randint(0, diffusion_obj.timesteps, (mri.shape[0],), device=config.device).long()
            loss, l1_loss, percep_loss = diffusion_obj.p_losses(ct, mri, t)
            val_loss_total += loss.item()
            val_l1_total += l1_loss.item()
            val_percep_total += percep_loss.item()

        n_batches = len(val_loader)
        avg_val_loss = val_loss_total / n_batches if n_batches > 0 else float('inf')
        avg_val_l1 = val_l1_total / n_batches if n_batches > 0 else float('inf')
        avg_val_percep = val_percep_total / n_batches if n_batches > 0 else float('inf')
        avg_psnr = np.mean([p for p in psnr_list if not np.isnan(p)]) if psnr_list else 0.0
        avg_ssim = np.mean([s for s in ssim_list if not np.isnan(s)]) if ssim_list else 0.0

    except StopIteration:
         print("Warning: Validation loader is empty.")
         avg_val_loss, avg_psnr, avg_ssim, avg_val_l1, avg_val_percep = float('inf'), 0.0, 0.0, float('inf'), float('inf')
    except Exception as e:
         print(f"Error during validation: {e}")
         avg_val_loss, avg_psnr, avg_ssim, avg_val_l1, avg_val_percep = float('inf'), 0.0, 0.0, float('inf'), float('inf')

    model.train()
    return avg_val_loss, avg_psnr, avg_ssim, avg_val_l1, avg_val_percep

print(" Validation function defined.")

In [ ]:

# Training

print("\n Setting up training components (Optimizer, Cosine Scheduler, Scaler)")
optimizer = optim.AdamW(model.parameters(), lr=config.learning_rate,
                        weight_decay=config.weight_decay, betas=(0.9, 0.999))
                        
def get_lr_lambda(epoch):
    warmup_epochs = max(1, config.warmup_epochs)
    total_epochs = max(1, config.epochs)
    if epoch < warmup_epochs:
        return float(epoch + 1) / float(warmup_epochs)
    else:
        progress = float(epoch - warmup_epochs) / float(max(1, total_epochs - warmup_epochs))
        return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=get_lr_lambda)
print(" Using Cosine Annealing scheduler with warmup.")

scaler = GradScaler() if config.use_amp else None
if config.use_amp: print(" AMP Scaler enabled.")
else: print(" AMP disabled.")

print(" Training setup complete.")

In [ ]:

# Training

print("\n Starting FULL Training Loop")

if config.resume_checkpoint and os.path.exists(config.resume_checkpoint):
    print(f" Resuming Training from: {config.resume_checkpoint}")
    try:
        checkpoint = torch.load(config.resume_checkpoint, map_location=config.device, weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        
        if scaler and 'scaler_state_dict' in checkpoint and checkpoint['scaler_state_dict'] is not None:
            scaler.load_state_dict(checkpoint['scaler_state_dict'])
            
        if ema and 'ema_shadow' in checkpoint and checkpoint['ema_shadow'] is not None:
            ema.shadow = checkpoint['ema_shadow']
            
        config.start_epoch = checkpoint['epoch'] + 1
        best_psnr = checkpoint.get('best_psnr', 0.0) 
        
        print(f"  Resumed model state from epoch {checkpoint['epoch']}")
        print(f"  Resumed optimizer and scheduler")
        print(f"  Resumed EMA and Scaler state")
        print(f"  Previous Best PSNR: {best_psnr:.2f} dB")
        print(f"  Starting training from Epoch {config.start_epoch}")
  
    except Exception as e:
        print(f"! ERROR loading checkpoint: {e}. Starting from scratch. !")
        config.start_epoch = 1
        best_psnr = 0.0
else:
    print(f"Checkpoint not specified or found. Starting Training from Scratch (Epoch 1)")
    config.start_epoch = 1
    best_psnr = 0.0 
if not train_loader:
    print("!!! FATAL ERROR: train_loader is not defined. Cannot start training loop.")
    print("!!! Please check errors.")
else:
    print(f" STARTING FULL TRAINING ({config.epochs} Epochs on {config.max_train_samples} samples)")
    print(f"Target PSNR: > 25 dB")
    print(f"Using Loss: L1(noise) + L1(image)")
    print(f"Attention Res: {config.attention_resolutions}")


    train_losses, val_losses, psnr_history, ssim_history, learning_rates = [], [], [], [], []
    l1_losses_train, percep_losses_train = [], [] 
    l1_losses_val, percep_losses_val = [], []
    
    start_time = datetime.now()

    for epoch in range(config.start_epoch, config.epochs + 1):
        model.train()
        epoch_train_loss, epoch_l1_loss, epoch_percep_loss = 0.0, 0.0, 0.0
        progress_bar = tqdm(train_loader, desc=f'Epoch {epoch}/{config.epochs}', leave=False)

        for batch_idx, batch in enumerate(progress_bar):
            if 'mri' not in batch or 'ct' not in batch: continue
            mri = batch['mri'].to(config.device)
            ct = batch['ct'].to(config.device)
            if mri.shape[-2:] != (config.img_size, config.img_size) or ct.shape[-2:] != (config.img_size, config.img_size): continue
            if mri.abs().max() < 1e-6 or ct.abs().max() < 1e-6: continue

            t = torch.randint(0, diffusion.timesteps, (mri.shape[0],), device=config.device).long()
            optimizer.zero_grad(set_to_none=True)

            if config.use_amp and scaler:
                with autocast():
                    loss, l1_loss, percep_loss = diffusion.p_losses(ct, mri, t)
                scaler.scale(loss).backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss, l1_loss, percep_loss = diffusion.p_losses(ct, mri, t)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
                optimizer.step()

            if config.use_ema and ema: ema.update()

            loss_item = loss.item()
            if not math.isnan(loss_item):
                 epoch_train_loss += loss_item
                 epoch_l1_loss += l1_loss.item()     
                 epoch_percep_loss += percep_loss.item() 

            progress_bar.set_postfix({
                'loss': f'{loss_item:.4f}', 'L1_noise': f'{l1_loss.item():.4f}', 'L1_image': f'{percep_loss.item():.4f}',
                'lr': f'{optimizer.param_groups[0]["lr"]:.6f}'
            })

        n_train_batches = len(train_loader)
        avg_train_loss = epoch_train_loss / n_train_batches if n_train_batches > 0 else 0
        avg_l1_loss = epoch_l1_loss / n_train_batches if n_train_batches > 0 else 0
        avg_percep_loss = epoch_percep_loss / n_train_batches if n_train_batches > 0 else 0
        train_losses.append(avg_train_loss)
        l1_losses_train.append(avg_l1_loss)
        percep_losses_train.append(avg_percep_loss)
        learning_rates.append(optimizer.param_groups[0]['lr'])
        
        scheduler.step()
        if epoch % config.val_interval == 0:
            current_ema = None
            if config.use_ema and ema:
                current_ema = ema
                current_ema.apply_shadow()

            avg_val_loss, avg_psnr, avg_ssim, avg_val_l1, avg_val_percep = validate_epoch(
                model, diffusion, val_loader, epoch
            )
            
            if current_ema: current_ema.restore()

            if val_loader:
                val_losses.append(avg_val_loss)
                l1_losses_val.append(avg_val_l1)
                percep_losses_val.append(avg_val_percep)
                psnr_history.append(avg_psnr)
                ssim_history.append(avg_ssim)
                print(f"\n Validation Epoch {epoch}")
                print(f"  Avg Val Loss (Combined): {avg_val_loss:.4f}")
                print(f"  Avg Val L1_noise Loss  : {avg_val_l1:.4f}")
                print(f"  Avg Val L1_image Loss  : {avg_val_percep:.4f}")
                print(f"  Avg PSNR (samples)     : {avg_psnr:.2f} dB")
                print(f"  Avg SSIM (samples)     : {avg_ssim:.4f}")

                if avg_psnr > best_psnr:
                    best_psnr = avg_psnr
                    print(f"  Saving new best model with PSNR: {best_psnr:.2f} dB")
                    save_path = os.path.join(config.output_dir, 'best_model_psnr_full.pt')
                    try:
                        torch.save({
                            'epoch': epoch, 'model_state_dict': model.state_dict(),
                            'psnr': avg_psnr, 'ssim': avg_ssim,
                            'ema_shadow': ema.shadow if config.use_ema and ema else None,
                        }, save_path)
                        print(f"   Model saved to {save_path}")
                    except Exception as e: print(f"  ERROR saving best model: {e}")
            

        if epoch % config.sample_interval == 0 and val_loader:
            print(f" Generating Samples Epoch {epoch}")
            current_ema = None
            if config.use_ema and ema:
                current_ema = ema
                current_ema.apply_shadow()
            model.eval()
            with torch.no_grad():
                try:
                    batch = next(iter(val_loader))
                    mri = batch['mri'][:config.num_sample_images].to(config.device)
                    ct_real = batch['ct'][:config.num_sample_images].to(config.device)
                    ct_gen = diffusion.ddim_sample(mri, num_steps=config.ddim_steps)

                    fig, axes = plt.subplots(config.num_sample_images, 3, figsize=(9, 3 * config.num_sample_images))
                    if config.num_sample_images == 1: axes = axes.reshape(1, -1)
                    for i in range(config.num_sample_images):
                        axes[i, 0].imshow((mri[i, 0].cpu().numpy() + 1) / 2, cmap='gray', vmin=0, vmax=1); axes[i, 0].axis('off')
                        axes[i, 1].imshow((ct_real[i, 0].cpu().numpy() + 1) / 2, cmap='gray', vmin=0, vmax=1); axes[i, 1].axis('off')
                        axes[i, 2].imshow((ct_gen[i, 0].cpu().numpy() + 1) / 2, cmap='gray', vmin=0, vmax=1); axes[i, 2].axis('off')
                        if i == 0: axes[i, 0].set_title('MRI Input'); axes[i, 1].set_title('Real CT'); axes[i, 2].set_title('Generated CT')
                    plt.tight_layout()
                    sample_path = os.path.join(config.output_dir, 'samples', f'epoch_{epoch}_full.png')
                    plt.savefig(sample_path, dpi=100)
                    plt.close(fig)
                    print(f" Samples saved to {sample_path}")
                except StopIteration: print("Could not generate samples: Val loader empty.")
                except Exception as e: print(f"Error generating samples: {e}")
            model.train()
            if current_ema: current_ema.restore()
          
        if epoch % config.save_interval == 0:
             chkpt_path = os.path.join(config.output_dir, 'checkpoints', f'checkpoint_epoch_{epoch}_full.pt')
             print(f"Saving Checkpoint Epoch {epoch}")
             try:
                save_dict = {
                    'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(), 'scheduler_state_dict': scheduler.state_dict(),
                    'ema_shadow': ema.shadow if config.use_ema and ema else None,
                    'scaler_state_dict': scaler.state_dict() if scaler else None,
                    'best_psnr' : best_psnr
                }
                torch.save(save_dict, chkpt_path)
                print(f" Checkpoint saved to {chkpt_path}")
             except Exception as e: print(f"ERROR saving checkpoint: {e}")
        
        if epoch % 10 == 0: gc.collect(); torch.cuda.empty_cache()

    total_time = datetime.now() - start_time
    print(" FULL TRAINING COMPLETE!"); 
    print(f"Total time: {total_time}")
    print(f"Best Validation PSNR: {best_psnr:.2f} dB")
    print(f"Final train loss: {train_losses[-1]:.4f}" if train_losses else "N/A")
    if val_losses: print(f"Final validation loss: {val_losses[-1]:.4f}")
    
    history_path = os.path.join(config.output_dir, 'training_history_full.json')
    try:
        history_data = {
            'train_losses': [float(l) for l in train_losses], 'l1_losses_train': [float(l) for l in l1_losses_train],
            'percep_losses_train': [float(l) for l in percep_losses_train], 
            'val_losses': [float(l) for l in val_losses], 'l1_losses_val': [float(l) for l in l1_losses_val],
            'percep_losses_val': [float(l) for l in percep_losses_val],
            'psnr_history': [float(p) for p in psnr_history], 'ssim_history': [float(s) for s in ssim_history],
            'learning_rates': [float(lr) for lr in learning_rates],
            'best_psnr': float(best_psnr), 'total_time': str(total_time),
        }
        with open(history_path, 'w') as f: json.dump(history_data, f, indent=4)
        print(f" Training history saved to {history_path}")
    except Exception as e: print(f"Error saving training history: {e}")

    if psnr_history and val_losses:
        print("Plotting training curves...")
        try:
            fig, axes = plt.subplots(1, 3, figsize=(15, 4))
            
            epochs_axis = range(1, len(train_losses) + 1)
            val_epochs_axis = [i for i in range(config.val_interval, len(train_losses) + 1, config.val_interval)]
            if len(val_epochs_axis) > len(val_losses): 
                val_epochs_axis = val_epochs_axis[:len(val_losses)]
            
            axes[0].plot(epochs_axis, train_losses, label='Train Loss (Total)', lw=2)
            axes[0].plot(epochs_axis, l1_losses_train, label='Train Loss (L1_noise)', lw=1, linestyle='--') 
            axes[0].plot(epochs_axis, percep_losses_train, label='Train Loss (L1_image)', lw=1, linestyle=':') 
            
            if len(val_epochs_axis) == len(val_losses):
                 axes[0].plot(val_epochs_axis, val_losses, label='Val Loss (Combined)', lw=2, marker='o', markersize=4)
                 axes[0].plot(val_epochs_axis, l1_losses_val, label='Val Loss (L1_noise)', lw=1, marker='x', markersize=4, linestyle='--')
                 axes[0].plot(val_epochs_axis, percep_losses_val, label='Val Loss (L1_image)', lw=1, marker='+', markersize=4, linestyle=':')
            else:
                 print(f"Warning: Mismatch plotting val_epochs ({len(val_epochs_axis)}) and val_losses ({len(val_losses)}). Skipping Val Loss plot.")
            
            axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].set_title('Loss')
            axes[0].legend(); axes[0].grid(alpha=0.3)

            if len(val_epochs_axis) == len(psnr_history):
                 axes[1].plot(val_epochs_axis, psnr_history, label='Val PSNR', lw=2, color='g', marker='o', markersize=4)
                 axes[1].axhline(y=best_psnr, color='r', linestyle='--', label=f'Best: {best_psnr:.2f} dB', lw=1)
            else:
                  print(f"Warning: MMismatch plotting val_epochs ({len(val_epochs_axis)}) and psnr_history ({len(psnr_history)}). Skipping PSNR plot.")
            axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('PSNR (dB)'); axes[1].set_title('PSNR')
            axes[1].legend(); axes[1].grid(alpha=0.3)
            
            axes[2].plot(epochs_axis, learning_rates, label='LR', lw=2, color='orange')
            axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Learning Rate'); axes[2].set_title('LR Schedule')
            axes[2].set_yscale('log'); axes[2].grid(alpha=0.3)

            plt.tight_layout()
            curves_path = os.path.join(config.output_dir, 'figures', 'training_curves_full.png')
            plt.savefig(curves_path, dpi=100)
            plt.show()
            plt.close(fig)
            print(f" Training curves saved to {curves_path}")
        except Exception as e:
            print(f"Error plotting training curves: {e}")


# END OF SCRIPT

print("\n End of Full Script ")

In [ ]:

# evaluation 

try:
    import lpips
except ImportError:
    print("Installing LPIPS library...")
    !pip install -q lpips
    import lpips

import torch
import numpy as np
import pandas as pd
import os
from tqdm.auto import tqdm
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
from scipy.stats import pearsonr


BEST_MODEL_PATH = "/kaggle/input/best-psnr-new/best_model_psnr_full (2).pt"
IS_DIFFUSION_MODEL = True   
IS_PIX2PIX_MODEL   = False

print(f"\n--- Setting up Evaluation for HYBRID DIFFUSION ---")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

loss_fn_alex = lpips.LPIPS(net='alex').to(device)
if os.path.exists(BEST_MODEL_PATH):
    print(f"Loading weights from: {BEST_MODEL_PATH}")

    checkpoint = torch.load(BEST_MODEL_PATH, map_location=device, weights_only=False)
    
    if IS_DIFFUSION_MODEL:
        if 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'])
        elif 'state_dict' in checkpoint:
            model.load_state_dict(checkpoint['state_dict'])
        else:
            model.load_state_dict(checkpoint)
        model.eval()
        
    print(" Best model weights loaded successfully.")
else:
    print(f" FATAL ERROR: File not found at {BEST_MODEL_PATH}")
    print("Please check the path in the 'Data' sidebar.")

def calculate_single_metrics(real_im, gen_im, real_tensor, gen_tensor):
    m = {}
    m['MAE'] = np.mean(np.abs(real_im - gen_im))
    m['RMSE'] = np.sqrt(np.mean((real_im - gen_im) ** 2))
    m['PSNR'] = psnr(real_im, gen_im, data_range=1.0)
    m['SSIM'] = ssim(real_im, gen_im, data_range=1.0, win_size=7, channel_axis=None)
    pcc, _ = pearsonr(real_im.flatten(), gen_im.flatten())
    m['PCC'] = pcc
    with torch.no_grad():
       
        lpips_val = loss_fn_alex(real_tensor.repeat(1,3,1,1), gen_tensor.repeat(1,3,1,1))
    m['LPIPS'] = lpips_val.item()
    return m

print(f"\nStarting evaluation on {len(val_loader)} batches...")
metric_history = {'MAE': [], 'RMSE': [], 'PSNR': [], 'SSIM': [], 'PCC': [], 'LPIPS': []}

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Evaluating"):
        mri = batch['mri'].to(device)
        real_ct = batch['ct'].to(device)
        fake_ct = diffusion.ddim_sample(mri, num_steps=50)
        
        for i in range(mri.shape[0]):
            real_t = real_ct[i:i+1]
            fake_t = fake_ct[i:i+1]
            real_np = ((real_t.cpu().numpy().squeeze() + 1) / 2.0).clip(0, 1)
            fake_np = ((fake_t.cpu().numpy().squeeze() + 1) / 2.0).clip(0, 1)
            
            res = calculate_single_metrics(real_np, fake_np, real_t, fake_t)
            for key in metric_history:
                metric_history[key].append(res[key])

df = pd.DataFrame(metric_history)
print("  HYBRID MODEL EVALUATION REPORT (Mean ± Std Dev)")
print(f"{'Metric':<10} | {'Mean':<10} | {'Std Dev':<10} | {'Interpretation'}")
better_low = ['MAE', 'RMSE', 'LPIPS']
for col in df.columns:
    mean_val = df[col].mean()
    std_val = df[col].std()
    direction = "↓ (Low is better)" if col in better_low else "↑ (High is better)"
    print(f"{col:<10} | {mean_val:<10.4f} | {std_val:<10.4f} | {direction}")


In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

os.makedirs("paper_figures", exist_ok=True)
print("Generating Figure 1: Visual Samples & Error Maps...")

def plot_visual_comparison(mri, real, fake, filename_prefix):
    error_map = np.abs(real - fake)
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
  
    axes[0].imshow(mri, cmap='gray', vmin=0, vmax=1)
    axes[0].set_title("Input MRI")
    axes[0].axis('off')
    
    axes[1].imshow(real, cmap='gray', vmin=0, vmax=1)
    axes[1].set_title("Ground Truth CT")
    axes[1].axis('off')
    
    axes[2].imshow(fake, cmap='gray', vmin=0, vmax=1)
    axes[2].set_title(f"Generated CT\n({filename_prefix})")
    axes[2].axis('off')
    

    im = axes[3].imshow(error_map, cmap='inferno', vmin=0, vmax=0.5) 
    axes[3].set_title("Absolute Error Map")
    axes[3].axis('off')
    
    plt.colorbar(im, ax=axes[3], fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.savefig(f"paper_figures/{filename_prefix}_visual_comparison.png", dpi=300)
    plt.show()

with torch.no_grad():
    batch = next(iter(val_loader))
    mri_viz = batch['mri'].to(config.device)
    real_viz = batch['ct'].to(config.device)
  
    if 'diffusion' in globals():
        fake_viz = diffusion.ddim_sample(mri_viz, num_steps=50)
    elif 'gen' in globals():
        fake_viz = gen(mri_viz)

    m_np = (mri_viz[0, 0].cpu().numpy() + 1) / 2.0
    r_np = (real_viz[0, 0].cpu().numpy() + 1) / 2.0
    f_np = (fake_viz[0, 0].cpu().numpy() + 1) / 2.0

    plot_visual_comparison(m_np, r_np, f_np, "Sample_1")


print("\nGenerating Figure 2: Metric Distributions...")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.boxplot(data=df, y='PSNR', ax=axes[0], color='skyblue')
axes[0].set_title("PSNR Distribution (Higher is Better)")
sns.boxplot(data=df, y='SSIM', ax=axes[1], color='lightgreen')
axes[1].set_title("SSIM Distribution (Higher is Better)")
sns.boxplot(data=df, y='MAE', ax=axes[2], color='salmon')
axes[2].set_title("MAE Distribution (Lower is Better)")

plt.tight_layout()
plt.savefig("paper_figures/metric_boxplots.png", dpi=300)
plt.show()


print("\nGenerating Figure 3: Intensity Scatter Plot...")

real_pixels = r_np.flatten()
fake_pixels = f_np.flatten()
idx = np.random.choice(len(real_pixels), 5000, replace=False)

plt.figure(figsize=(6, 6))
plt.scatter(real_pixels[idx], fake_pixels[idx], alpha=0.1, s=1, c='blue')
plt.plot([0, 1], [0, 1], 'r--', label="Ideal (y=x)") 
plt.xlabel("Ground Truth Pixel Intensity")
plt.ylabel("Generated Pixel Intensity")
plt.title("Pixel Intensity Correlation")
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig("paper_figures/scatter_correlation.png", dpi=300)
plt.show()
